In [ ]:
import os

# Must come before any torch / habana import.
# Enables HPU eager mode when using standard PyTorch (not Habana's fork).
# No effect on CPU or CUDA. Override with PT_HPU_LAZY_MODE=1 in your
# shell environment if you have Habana's lazy-mode torch fork installed.
if "PT_HPU_LAZY_MODE" not in os.environ:
    os.environ["PT_HPU_LAZY_MODE"] = "0"

# On a shared multi-HPU machine, each card is exclusive to one process — if
# this kernel and another job end up on the same card, Habana's runtime can
# hard-crash the loser (shows up here as "kernel crashed", no traceback).
# Run `hl-smi` in a terminal first to see which card is actually idle, then
# pin to it. No effect on CPU or CUDA.
if "HABANA_VISIBLE_MODULES" not in os.environ:
    os.environ["HABANA_VISIBLE_MODULES"] = "1"

# LLM Membership Inference Attack (EZ-MIA)

This runs `leakpro.llm_attacks.mia`, a standalone error-zone MIA attack for causal LLMs. Unlike the other attacks under `examples/mia/`, it does **not** go through `Leakpro(...)`/`audit.yaml`: it trains its own target LLM and a reference model as part of the experiment, so it's called directly via `run_attack(cfg)` instead. See `leakpro/llm_attacks/mia/README.md` for details.

In [ ]:
import sys


def find_project_root(marker='pyproject.toml'):
    current = os.path.abspath(os.getcwd())
    while True:
        if os.path.exists(os.path.join(current, marker)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError(f"Project root (containing {marker}) not found")
        current = parent


project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from leakpro.llm_attacks.mia.attack import run_attack
from leakpro.llm_attacks.mia.config import AttackConfig, load_attack_config_from_yaml
from leakpro.utils.device import get_device

print(f"Using device: {get_device()}")

## Quick smoke-test run

A tiny GPT-2 + AG News + base-reference run, sized to finish quickly so you can confirm the pipeline works end to end on this machine (CPU, CUDA, or HPU). Not enough data for a meaningful AUC — for that, use the YAML-config run below.

In [ ]:
smoke_cfg = AttackConfig(
    dataset="ag_news",
    ref_variant="base",
    model_name="gpt2",
    seed=42,
    train_total=200,
    eval_total=100,
    val_total=50,
    epochs=1,
    batch_size=8,
    sequence_length=64,
)

smoke_result = run_attack(smoke_cfg)
smoke_result

## Full experiment from a YAML config

Uses one of the example configs shipped in `leakpro/llm_attacks/mia/configs/`. This trains on far more data and will take substantially longer than the smoke test above.

In [ ]:
config_path = os.path.join(project_root, "leakpro/llm_attacks/mia/configs/exp_ag_news_gpt2_base.yaml")
cfg = load_attack_config_from_yaml(config_path)

result = run_attack(cfg)
result

`result` reports `auc`, `tpr_at_fpr_0.01`, and `tpr_at_fpr_0.001` for the run. The CLI (`python -m leakpro.llm_attacks.mia --config ...`) additionally appends each run to `leakpro/llm_attacks/mia/results.csv`; calling `run_attack` directly, as done here, does not.